# Silver — Arquitectura Medallón con Datos de Fraude

**Semana:** 02  
**Actividad:** 04  
**Capa:** Silver  
**Notebook:** silver_daniel  

## Objetivo

Leer siempre desde las tablas Bronze, limpiar tipos de datos, estandarizar nombres de columnas, preparar las llaves de JOIN y construir una tabla maestra enriquecida para análisis de fraude.

## Entrada

- `workspace.bronze.transactions_daniel`
- `workspace.bronze.users_daniel`
- `workspace.bronze.cards_daniel`
- `workspace.bronze.mcc_codes_daniel`
- `workspace.bronze.fraud_labels_daniel`

## Salida

- `workspace.silver.transactions_daniel`

## Transformaciones principales

- Convertir `amount` a tipo numérico.
- Convertir `date` a timestamp.
- Crear columnas temporales: hora, día de semana, mes y año.
- Renombrar llaves para hacer JOINs.
- Convertir MCC de formato ancho a formato largo.
- Unir transactions, users, cards, MCC codes y fraud labels.
- Crear columna `is_fraud` numérica para análisis.
- Validar duplicados por `transaction_id`.

In [0]:
from pyspark.sql import functions as F

MI_NOMBRE = "daniel"
CATALOG = "workspace"

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")

# Leer SIEMPRE desde Bronze
df_tx = spark.table(f"{CATALOG}.bronze.transactions_{MI_NOMBRE}")
df_users = spark.table(f"{CATALOG}.bronze.users_{MI_NOMBRE}")
df_cards = spark.table(f"{CATALOG}.bronze.cards_{MI_NOMBRE}")
df_mcc_raw = spark.table(f"{CATALOG}.bronze.mcc_codes_{MI_NOMBRE}")
df_fraud = spark.table(f"{CATALOG}.bronze.fraud_labels_{MI_NOMBRE}")

print("Tablas Bronze cargadas correctamente")
print(f"transactions: {df_tx.count():,}")
print(f"users: {df_users.count():,}")
print(f"cards: {df_cards.count():,}")
print(f"mcc raw: {df_mcc_raw.count():,}")
print(f"fraud labels: {df_fraud.count():,}")

In [0]:
import re

def to_snake_case(col_name):
    col_name = col_name.strip().lower()
    col_name = re.sub(r"[^a-z0-9]+", "_", col_name)
    col_name = re.sub(r"_+", "_", col_name).strip("_")
    return col_name

def normalize_columns(df):
    for old_col in df.columns:
        df = df.withColumnRenamed(old_col, to_snake_case(old_col))
    return df

df_tx = normalize_columns(df_tx)
df_users = normalize_columns(df_users)
df_cards = normalize_columns(df_cards)
df_fraud = normalize_columns(df_fraud)

print("Columnas transactions:")
print(df_tx.columns)

print("Columnas users:")
print(df_users.columns)

print("Columnas cards:")
print(df_cards.columns)

print("Columnas fraud:")
print(df_fraud.columns)

In [0]:
df_tx_clean = (
    df_tx
    .withColumnRenamed("id", "transaction_id")
    .withColumnRenamed("client_id", "user_id")
    .withColumn("amount", F.regexp_replace(F.col("amount"), "[$,]", "").cast("double"))
    .withColumn("amount_abs", F.abs(F.col("amount")))
    .withColumn("transaction_date", F.to_timestamp(F.col("date"), "yyyy-MM-dd HH:mm:ss"))
    .withColumn("hora", F.hour("transaction_date"))
    .withColumn("dia_semana", F.dayofweek("transaction_date"))
    .withColumn("es_fin_de_semana", F.when(F.col("dia_semana").isin([1, 7]), 1).otherwise(0))
    .withColumn("mes", F.month("transaction_date"))
    .withColumn("anio", F.year("transaction_date"))
    .withColumn("mcc", F.col("mcc").cast("int"))
    .drop("date")
)

df_tx_clean.printSchema()
display(df_tx_clean.limit(5))

In [0]:
df_users_clean = (
    df_users
    .withColumnRenamed("id", "user_id")
)

df_cards_clean = (
    df_cards
    .withColumnRenamed("id", "card_id")
    .withColumnRenamed("client_id", "card_user_id")
)

print("Users limpio:")
df_users_clean.printSchema()

print("Cards limpio:")
df_cards_clean.printSchema()

In [0]:
mcc_cols = df_mcc_raw.columns

stack_expr = (
    f"stack({len(mcc_cols)}, "
    + ", ".join([f"'{c}', `{c}`" for c in mcc_cols])
    + ") as (mcc_str, merchant_category)"
)

df_mcc_clean = (
    df_mcc_raw
    .select(F.expr(stack_expr))
    .withColumn("mcc", F.col("mcc_str").cast("int"))
    .drop("mcc_str")
)

print(f"Categorías MCC: {df_mcc_clean.count():,}")
display(df_mcc_clean.limit(10))

In [0]:
# El archivo real trae la columna target con valores Yes / No
df_fraud_clean = (
    df_fraud
    .withColumnRenamed("id", "transaction_id")
    .withColumnRenamed("target", "is_fraud_label")
    .withColumn(
        "is_fraud",
        F.when(F.col("is_fraud_label") == "Yes", 1)
         .when(F.col("is_fraud_label") == "No", 0)
         .otherwise(None)
    )
)

df_fraud_clean.printSchema()
display(df_fraud_clean.limit(5))

In [0]:
df_silver = (
    df_tx_clean
    .join(df_users_clean, on="user_id", how="left")
    .join(df_cards_clean, on="card_id", how="left")
    .join(df_mcc_clean, on="mcc", how="left")
    .join(df_fraud_clean, on="transaction_id", how="left")
)

print(f"Filas Silver: {df_silver.count():,}")
print(f"Columnas Silver: {len(df_silver.columns)}")

display(df_silver.limit(5))

In [0]:
dup_count = (
    df_silver
    .groupBy("transaction_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

sin_usuario = df_silver.filter(F.col("user_id").isNull()).count()
sin_card_type = df_silver.filter(F.col("card_type").isNull()).count()
sin_categoria = df_silver.filter(F.col("merchant_category").isNull()).count()
sin_label = df_silver.filter(F.col("is_fraud").isNull()).count()

print(f"Transacciones duplicadas tras JOIN: {dup_count:,}")
print(f"Registros sin user_id: {sin_usuario:,}")
print(f"Registros sin card_type: {sin_card_type:,}")
print(f"Registros sin merchant_category: {sin_categoria:,}")
print(f"Registros sin etiqueta de fraude: {sin_label:,}")

In [0]:
df_silver.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG}.silver.transactions_{MI_NOMBRE}")

print(f"Tabla guardada: {CATALOG}.silver.transactions_{MI_NOMBRE}")
print(f"Filas: {df_silver.count():,}")
print(f"Columnas: {len(df_silver.columns)}")

## Documentación de cambios entre Bronze y Silver

En Bronze los datos se guardaron tal como llegaron desde las fuentes, sin inferir tipos en los CSV y sin aplicar limpieza.

En Silver se aplicaron transformaciones para dejar una tabla maestra enriquecida:

### Cambios principales

- `id` de transactions se renombró a `transaction_id`.
- `client_id` se renombró a `user_id`.
- `amount` se convirtió de string a double.
- `date` se convirtió a `transaction_date` tipo timestamp.
- Se crearon columnas temporales: `hora`, `dia_semana`, `es_fin_de_semana`, `mes` y `anio`.
- `mcc_codes` se convirtió de formato ancho a formato largo.
- `target` de fraud labels se convirtió en:
  - `is_fraud_label`: valor original Yes/No.
  - `is_fraud`: valor numérico 1/0 para agregaciones.
- Se unieron las 5 tablas Bronze en una tabla maestra.
- Se validaron duplicados por `transaction_id`.

### Columnas eliminadas o ajustadas

La columna original `date` se eliminó después de crear `transaction_date`, porque la nueva columna tiene el tipo correcto para análisis temporal.

No se realizaron agregaciones en Silver. Las agregaciones quedan para la capa Gold.